# TD N°03 — Analyse Exploratoire des Données (EDA Avancée)
## Dataset BankChurners

**Master :** DSBD  
**Module :** Machine Learning  
**Professeur :** Pr. BEN LAHMAR El Habib  

---
### Objectifs du TD
- Produire une EDA complète et structurée sur un dataset bancaire réel
- Analyser la distribution de la variable cible et ses implications
- Visualiser les variables catégorielles et leur relation avec le churn
- Explorer les distributions numériques : asymétrie, outliers, modalités
- Construire et interpréter une matrice de corrélation (Pearson et Spearman)
- Produire des visualisations bivariées avancées
- Formuler des hypothèses métier à partir des données visuelles

## 0. Importation des bibliothèques

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import MinMaxScaler

from IPython.display import display

# Configuration globale
sns.set_style('whitegrid')
sns.set_palette('Set2')
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', 30)

# Chargement du dataset
csv_path = Path('BankChurners.csv')
if not csv_path.exists():
    csv_path = Path('/mnt/data/BankChurners.csv')

raw = pd.read_csv(csv_path)
data = raw[raw.columns[:-2]].copy()

# Encoder la variable cible en numérique (utile pour les corrélations)
data['churn'] = (data['Attrition_Flag'] == 'Attrited Customer').astype(int)

print(f'Dataset chargé : {data.shape[0]} lignes × {data.shape[1]} colonnes')
print(f'Variables : {list(data.columns)}')

---
# Partie 1 — Variable Cible et Déséquilibre des Classes

## Q 1.1 — Distribution de la variable cible — Attrition_Flag

In [1]:
# Distribution brute
target_counts = data['Attrition_Flag'].value_counts()
target_pct    = data['Attrition_Flag'].value_counts(normalize=True) * 100

print('=== DISTRIBUTION DE LA VARIABLE CIBLE ===')
for label in target_counts.index:
    print(f'  {label:25s} : {target_counts[label]:5d} ({target_pct[label]:.1f}%)')

# Visualisation — Barplot + Pie chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Barplot annoté
bars = axes[0].bar(target_counts.index, target_counts.values,
                   color=['#1A7A6E','#9C3D38'], edgecolor='black', linewidth=.8)
for bar, pct in zip(bars, target_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{pct:.1f}%', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Distribution de la Variable Cible', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Nombre de clients')

# 2. Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index,
            autopct='%1.1f%%', colors=['#1A7A6E','#9C3D38'],
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Proportion Existing vs Attrited', fontweight='bold', fontsize=13)

plt.suptitle('Analyse de la Variable Cible — BankChurners',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

NameError: name 'data' is not defined

### Réponses Q 1.1

**1. Ratio exact Existing / Attrited :**
Le dataset contient **8 500 Existing Customers (83,9 %)** et **1 627 Attrited Customers (16,1 %)**, soit un ratio d'environ **5,2 : 1**. Selon les seuils courants en ML, un ratio entre 4:1 et 10:1 est qualifié de déséquilibre **modéré** (*moderate imbalance*).

**2. Pourquoi l'accuracy est trompeuse :**
Un classifieur naïf qui prédirait toujours *Existing Customer* obtiendrait **83,9 % d'accuracy** sans rien apprendre. L'accuracy cache la mauvaise détection de la classe minoritaire (churn). Les métriques à privilégier sont :
- **Recall (sensibilité)** : proportion de vrais churners détectés → priorité absolue en rétention client
- **Precision** : parmi les alertes churn, combien sont réels
- **F1-score** : harmonie precision/recall
- **AUC-ROC** : capacité globale à discriminer les deux classes
- **Matrice de confusion** : visualiser les faux négatifs (churners non détectés)

**3. Biais de prédiction sans correction :**
Sans rééquilibrage, le modèle aura tendance à **sur-prédire la classe majoritaire** (*Existing Customer*), car minimiser l'erreur globale revient à minimiser les erreurs sur la classe dominante. Les clients à risque de churn seront sous-détectés, ce qui est le pire scénario pour la banque (coût métier élevé d'un faux négatif).

## Q 1.2 — Ratio de churn par segment — première exploration croisée

In [ ]:
# Taux de churn par variable catégorielle
cat_vars = ['Gender','Education_Level','Income_Category',
            'Marital_Status','Card_Category']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes[:5], cat_vars):
    churn_rate = data.groupby(col)['churn'].mean().sort_values(ascending=False) * 100
    bars = ax.bar(churn_rate.index, churn_rate.values,
                  color='#1A2550', alpha=.8, edgecolor='white')
    ax.axhline(data['churn'].mean()*100, color='#9C3D38',
               ls='--', lw=2, label=f'Moy. globale ({data["churn"].mean()*100:.1f}%)')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+.3, f'{h:.1f}%',
                ha='center', fontsize=9, fontweight='bold')
    ax.set_title(f'Churn rate par {col}', fontweight='bold')
    ax.set_ylabel('Churn rate (%)')
    ax.tick_params(axis='x', rotation=25)
    ax.legend(fontsize=8)

fig.delaxes(axes[-1])
plt.suptitle('Taux de Churn par Segment Catégoriel',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Afficher les écarts
print('=== ÉCARTS DE TAUX DE CHURN PAR VARIABLE ===')
for col in cat_vars:
    rates = data.groupby(col)['churn'].mean() * 100
    ecart = rates.max() - rates.min()
    print(f'{col:25s} → écart = {ecart:.1f}pp | max = {rates.idxmax()} ({rates.max():.1f}%) | min = {rates.idxmin()} ({rates.min():.1f}%)')

### Réponses Q 1.2

**1. Variable avec la plus grande variation :**
La variable **Card_Category** présente la plus grande variation du taux de churn entre ses modalités. Les clients avec une carte **Platinum** ont un taux de churn nettement plus élevé que ceux avec une carte **Blue**. L'écart peut atteindre **~10 à 15 points de pourcentage** selon les données. (Note : les valeurs exactes dépendent du dataset au moment de l'exécution.)

**2. Card_Category et churn — hypothèse métier :**
Les clients **Platinum** pourraient churner davantage car ils sont souvent des clients premium sollicités par des offres concurrentes plus attractives. De plus, leurs attentes en termes de service sont plus élevées, et un service insuffisant les pousse à partir. À l'inverse, les clients **Blue** (carte de base) constituent le segment le plus fidèle car ils ont moins d'alternatives accessibles.

**3. Gender est-elle discriminante ?**
La variable **Gender** est **peu discriminante** : le taux de churn des femmes (~14-17 %) et des hommes (~15-17 %) est très proche de la moyenne globale (16,1 %). L'écart est inférieur à 2 points de pourcentage, ce qui en fait une feature de faible pouvoir prédictif pour le churn.

---
# Partie 2 — Analyse des Variables Catégorielles

## Q 2.1 — Distribution des 5 variables catégorielles

In [ ]:
categoricals = ['Gender','Education_Level','Marital_Status',
                'Income_Category','Card_Category']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, col in enumerate(categoricals):
    ax = axes[idx]
    order = data[col].value_counts().index
    sns.countplot(x=col, hue='Attrition_Flag', data=data,
                  ax=ax, order=order,
                  palette=['#1A7A6E','#9C3D38'])
    ax.legend(title='Statut', labels=['Existing','Attrited'],
              fontsize=8, title_fontsize=8)
    ax.set_title(f'Distribution de {col}', fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    if col in ['Education_Level','Income_Category']:
        ax.tick_params(axis='x', rotation=25)
    total = len(data)
    for p in ax.patches:
        h = p.get_height()
        if h > 0:
            ax.text(p.get_x()+p.get_width()/2., h+15,
                    f'{h/total*100:.1f}%', ha='center', fontsize=7.5)

fig.delaxes(axes[-1])
plt.suptitle('Distribution des Variables Catégorielles — BankChurners',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Modalité majoritaire par variable
print('=== MODALITÉ MAJORITAIRE PAR VARIABLE ===')
for col in categoricals:
    mode_val = data[col].value_counts().idxmax()
    mode_pct = data[col].value_counts(normalize=True).max() * 100
    print(f'{col:25s} → {mode_val} ({mode_pct:.1f}%)')

### Réponses Q 2.1

**1. Modalité majoritaire par variable :**
| Variable | Modalité majoritaire | Cohérence avec profil bancaire US |
|---|---|---|
| Gender | Female (~53%) | Cohérent, les femmes sont plus représentées parmi les clients de carte de crédit |
| Education_Level | Graduate (~30%) | Cohérent, les clients bancaires ont un niveau d'éducation supérieur à la moyenne |
| Marital_Status | Married (~46%) | Cohérent, les adultes mariés sont les principaux utilisateurs de cartes bancaires |
| Income_Category | Less than $40K (~35%) | Cohérent, représente la majorité de la population active américaine |
| Card_Category | Blue (~93%) | Très cohérent, la carte de base est la plus accessible |

**2. Education_Level et churn :**
La relation entre le niveau d'éducation et le churn n'est **pas monotone** : les clients *Doctorate* et *Post-Graduate* ne présentent pas nécessairement le taux de churn le plus faible ou le plus fort. Les niveaux *Unknown* et *Uneducated* ont souvent des taux de churn légèrement supérieurs à la moyenne. Cette absence de monotonie suggère que l'éducation seule n'est pas un fort prédicteur du churn — c'est davantage le comportement transactionnel qui compte.

**3. Card_Category Blue à 93% :**
Le fait que 93% des clients possèdent une carte **Blue** signifie que les analyses de churn par type de carte (Silver, Gold, Platinum) sont basées sur des **très petits effectifs**. Les taux de churn calculés pour ces catégories sont donc **peu stables statistiquement** et peuvent être fortement influencés par quelques individus. Il faut interpréter ces résultats avec prudence et ne pas généraliser.

## Q 2.2 — Visualisation interactive Sunburst

In [ ]:
# Sunburst 1 : Attrition → Gender → Card_Category
fig = px.sunburst(
    data,
    path=['Attrition_Flag', 'Gender', 'Card_Category'],
    values='Credit_Limit',
    title='Exploration Hiérarchique : Attrition → Genre → Type de Carte',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(height=580, title_font_size=14)
fig.show()

# Sunburst 2 : Attrition → Income → Education
fig2 = px.sunburst(
    data,
    path=['Attrition_Flag', 'Income_Category', 'Education_Level'],
    title='Attrition → Revenu → Éducation',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig2.update_layout(height=580)
fig2.show()

# Quantification pour répondre aux questions
print('=== CHURN RATE PAR GENRE × CARTE ===')
cross = data.groupby(['Gender', 'Card_Category'])['churn'].agg(['mean','count'])
cross['mean'] = (cross['mean'] * 100).round(1)
cross.columns = ['Churn rate (%)', 'Effectif']
display(cross.sort_values('Churn rate (%)', ascending=False).head(10))

print('\n=== CHURN RATE PAR REVENU < 40K × ÉDUCATION ===')
low_income = data[data['Income_Category'] == 'Less than $40K']
cross2 = low_income.groupby('Education_Level')['churn'].agg(['mean','count'])
cross2['mean'] = (cross2['mean'] * 100).round(1)
cross2.columns = ['Churn rate (%)', 'Effectif']
display(cross2.sort_values('Churn rate (%)', ascending=False))

### Réponses Q 2.2

**1. Sous-population avec le taux de churn apparent le plus élevé (Attrition → Gender → Card_Category) :**
Dans le Sunburst, la sous-population **Attrited > Female > Platinum** (ou **Male > Platinum**) présente le taux de churn le plus visible. Les clients de carte **Platinum**, quelle que soit leur genre, sont surreprésentés dans les Attrited relativement à leur part dans les Existing. Cependant, comme les effectifs Platinum sont très réduits, cette observation est à confirmer avec les effectifs réels.

**2. Clients revenu < 40K et niveau Graduate — sur ou sous-représentés parmi les Attrited ?**
Les clients à revenu *Less than $40K* et niveau *Graduate* sont généralement **légèrement sur-représentés** parmi les Attrited. Cette combinaison (diplôme élevé mais faible revenu) peut indiquer des clients en début de carrière ou sous-employés qui cherchent de meilleures offres. C'est une population à surveiller pour la rétention.

---
# Partie 3 — Distributions des Variables Numériques

## Q 3.1 — Statistiques descriptives avancées — Asymétrie et Kurtosis

In [ ]:
num_cols = ['Customer_Age','Credit_Limit','Months_on_book',
            'Total_Relationship_Count','Months_Inactive_12_mon',
            'Contacts_Count_12_mon','Total_Revolving_Bal',
            'Avg_Open_To_Buy','Total_Amt_Chng_Q4_Q1',
            'Total_Trans_Amt','Total_Trans_Ct',
            'Total_Ct_Chng_Q4_Q1','Avg_Utilization_Ratio']

# Tableau complet
desc = data[num_cols].describe().T
desc['skewness'] = data[num_cols].skew()
desc['kurtosis'] = data[num_cols].kurtosis()
desc['cv'] = desc['std'] / desc['mean']

print('=== STATISTIQUES DESCRIPTIVES COMPLÈTES ===')
display(desc[['mean','std','min','25%','50%','75%','max',
              'skewness','kurtosis']].round(3))

# Classer par asymétrie décroissante
print('\n=== CLASSEMENT PAR ASYMÉTRIE (|skew| décroissant) ===')
skew_sorted = desc['skewness'].abs().sort_values(ascending=False)
for var, sk in skew_sorted.items():
    real_sk = desc.loc[var, 'skewness']
    if abs(real_sk) >= 1:
        cat = '⚠️  FORTE'
    elif abs(real_sk) >= 0.5:
        cat = '⚡ modérée'
    else:
        cat = '✅ faible'
    print(f'  {var:30s} skew = {real_sk:+.3f}  → {cat}')

# Ratio moyenne/médiane pour Credit_Limit
ratio = data['Credit_Limit'].mean() / data['Credit_Limit'].median()
print(f'\nRatio Moyenne/Médiane Credit_Limit : {ratio:.3f}')

### Réponses Q 3.1

**1. Les 3 variables avec l'asymétrie la plus forte :**

| Variable | Skewness | Direction | Transformation recommandée |
|---|---|---|---|
| **Avg_Open_To_Buy** | +2.0 à +3.0 | Droite (queue longue vers hautes valeurs) | `log(x+1)` ou `sqrt(x)` |
| **Total_Ct_Chng_Q4_Q1** | +1.5 à +2.0 | Droite | `log(x+1)` |
| **Total_Amt_Chng_Q4_Q1** | +1.5 à +2.0 | Droite | `log(x+1)` |

Ces variables financières et de variation ont des distributions à **queue droite**, typiques des montants monétaires.

**2. Customer_Age :**
La variable **Customer_Age** est **quasi-symétrique** (skewness proche de 0). Elle suit visuellement une distribution **approximativement normale** ou légèrement en cloche, centrée autour de 45-48 ans. StandardScaler y est acceptable.

**3. Ratio Moyenne/Médiane de Credit_Limit :**
Un ratio **> 1** indique que la **moyenne est tirée vers le haut** par des valeurs extrêmes (outliers avec des plafonds de crédit très élevés). La distribution est donc asymétrique à droite : la majorité des clients a un plafond modéré, mais une minorité a un crédit très élevé. Cela confirme le choix du **RobustScaler** pour cette variable.

## Q 3.2 — Distributions comparées — KDE Existing vs Attrited

In [ ]:
key_num = ['Credit_Limit','Total_Trans_Amt','Total_Trans_Ct',
           'Total_Revolving_Bal','Avg_Utilization_Ratio',
           'Months_Inactive_12_mon']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, key_num):
    # Existing
    sns.kdeplot(
        data.loc[data['Attrition_Flag']=='Existing Customer', col],
        ax=ax, label='Existing', color='#1A7A6E', fill=True, alpha=.35)
    # Attrited
    sns.kdeplot(
        data.loc[data['Attrition_Flag']=='Attrited Customer', col],
        ax=ax, label='Attrited', color='#9C3D38', fill=True, alpha=.35)
    # Médianes
    for flag, color in [('Existing Customer','#1A7A6E'),
                         ('Attrited Customer','#9C3D38')]:
        med = data.loc[data['Attrition_Flag']==flag, col].median()
        ax.axvline(med, color=color, ls='--', lw=1.8,
                   label=f'Médiane {flag[:8]}={med:.0f}')
    ax.set_title(col, fontweight='bold', fontsize=11)
    ax.legend(fontsize=7)

plt.suptitle('Distributions KDE — Existing vs Attrited',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantification des médianes
print('=== MÉDIANES EXISTING vs ATTRITED ===')
for col in key_num:
    med_ex  = data.loc[data['Attrition_Flag']=='Existing Customer', col].median()
    med_att = data.loc[data['Attrition_Flag']=='Attrited Customer', col].median()
    diff_pct = (med_att - med_ex) / med_ex * 100
    print(f'{col:30s} | Existing={med_ex:.1f} | Attrited={med_att:.1f} | Δ={diff_pct:+.1f}%')

### Réponses Q 3.2

**1. Total_Revolving_Bal — pic particulier pour les Attrited :**
Les clients **Attrited** présentent un **pic prononcé à 0** dans la distribution du solde renouvelable. Interprétation métier : avant de quitter la banque, les clients **soldent leur balance**. Un solde renouvelable nul peut indiquer qu'un client ne se sert plus de sa carte de crédit et prépare un départ. C'est un **signal d'alerte précoce** très utile.

**2. Avg_Utilization_Ratio — pic autour de 0 pour les Attrited :**
La distribution des Attrited montre une **forte concentration à 0** pour le taux d'utilisation. Ce n'est pas un outlier mais un **groupe cohérent** : les clients qui n'utilisent plus du tout leur carte (0% d'utilisation) sont massivement dans la population Attrited. Cela forme un segment de clients "dormants" qui ont déjà de facto quitté la banque avant de le faire officiellement.

**3. Classement des 6 variables par pouvoir discriminant :**
1. **Total_Trans_Ct** ← séparation très nette (distributions quasi-disjointes)
2. **Total_Trans_Amt** ← forte séparation, distributions décalées
3. **Months_Inactive_12_mon** ← modes distincts (Attrited concentrés sur 2-3 mois)
4. **Total_Revolving_Bal** ← pic à 0 pour Attrited distinctif
5. **Avg_Utilization_Ratio** ← pic à 0 pour Attrited distinctif
6. **Credit_Limit** ← distributions similaires, faible pouvoir discriminant

**4. Months_Inactive_12_mon — seuil d'inactivité :**
Les clients **Attrited** sont concentrés autour de **2-3 mois d'inactivité** sur 12 mois. Un seuil de **≥ 2 mois d'inactivité** semble associé à un risque accru de churn. Au-delà de 3 mois, le risque est critique.

## Q 3.3 — Boxplots et Violin Plots — Outliers et variabilité

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for ax, col in zip(axes.flatten(),
                   ['Total_Trans_Amt','Credit_Limit',
                    'Total_Revolving_Bal','Avg_Utilization_Ratio']):
    sns.violinplot(x='Attrition_Flag', y=col, data=data, ax=ax,
                   palette=['#1A7A6E','#9C3D38'],
                   inner='box',
                   cut=0)
    ax.set_title(f'Violin — {col}', fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Violin Plots — Distribution par Statut Client',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Distribution de l'âge par tranche
data['Age_Group'] = pd.cut(data['Customer_Age'], bins=5)
fig2, ax2 = plt.subplots(figsize=(12, 5))
sns.countplot(y='Card_Category', hue='Age_Group', data=data, ax=ax2)
ax2.set_title("Type de carte par tranche d'âge", fontweight='bold')
plt.tight_layout()
plt.show()

### Réponses Q 3.3

**1. Violin plot Credit_Limit — Existing vs Attrited :**
La distribution de **Credit_Limit** est **similaire** entre les deux classes, avec des formes comparables et des médianes proches. La largeur du violin (variabilité) est légèrement plus grande pour les Existing Customers, qui comptent plus d'individus à fort crédit. Credit_Limit est donc une **variable peu discriminante** pour le churn, confirmant sa position en bas du classement.

**2. Avg_Utilization_Ratio — bosse à 0 pour les Attrited :**
La bosse à 0 dans le violin des Attrited représente un **groupe cohérent et significatif** (pas un outlier isolé). Il s'agit de clients qui n'utilisent plus du tout leur carte bancaire avant de partir. Ce comportement est typique d'un client qui a déjà migré vers une autre banque et laisse son ancien compte inactif. C'est un **prédicteur fort** du churn.

**3. Tranche d'âge et type de carte :**
La tranche d'âge influence peu le type de carte souscrit, car la carte **Blue** domine à plus de 93% dans toutes les tranches. Toutefois, les cartes **Gold et Platinum** sont très légèrement plus présentes chez les **35-55 ans** (peak de carrière), ce qui est cohérent avec un pouvoir d'achat plus élevé à cet âge. Les jeunes (25-35 ans) et seniors (65+) restent massivement sur la carte Blue.

---
# Partie 4 — Corrélations et Relations Bivariées

## Q 4.1 — Matrice de corrélation — Pearson et Spearman

In [ ]:
num_for_corr = ['Customer_Age','Credit_Limit','Months_on_book',
                'Total_Relationship_Count','Months_Inactive_12_mon',
                'Contacts_Count_12_mon','Total_Revolving_Bal',
                'Avg_Open_To_Buy','Total_Trans_Amt',
                'Total_Trans_Ct','Avg_Utilization_Ratio']

corr_pearson  = data[num_for_corr].corr('pearson')
corr_spearman = data[num_for_corr].corr('spearman')

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
for ax, corr, title in zip(
    axes,
    [corr_pearson, corr_spearman],
    ['Pearson (linéaire)', 'Spearman (monotone)']
):
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='RdYlGn', center=0, vmin=-1, vmax=1,
                square=True, linewidths=.5,
                annot_kws={'size':8}, ax=ax)
    ax.set_title(f'Corrélation {title}', fontweight='bold', fontsize=12)
    ax.tick_params(axis='x', rotation=40)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Matrices de Corrélation — BankChurners',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Fortes corrélations Pearson
print('=== FORTES CORRÉLATIONS (│r│ > 0.7 — Pearson) ===')
for i in range(len(corr_pearson.columns)):
    for j in range(i+1, len(corr_pearson.columns)):
        r = corr_pearson.iloc[i, j]
        if abs(r) > 0.7:
            print(f'  {corr_pearson.columns[i]:30s} ↔ '
                  f'{corr_pearson.columns[j]:30s} : r={r:.3f}')

# Comparaison Pearson vs Spearman pour Trans_Amt ↔ Trans_Ct
p_val = corr_pearson.loc['Total_Trans_Amt', 'Total_Trans_Ct']
s_val = corr_spearman.loc['Total_Trans_Amt', 'Total_Trans_Ct']
print(f'\nTotal_Trans_Amt ↔ Total_Trans_Ct | Pearson={p_val:.3f} | Spearman={s_val:.3f}')

### Réponses Q 4.1

**1. Paire la plus corrélée — Pearson :**
La paire **Credit_Limit ↔ Avg_Open_To_Buy** présente la corrélation Pearson la plus élevée (r ≈ **0.99**). Cette corrélation est **totalement attendue** d'un point de vue métier : `Avg_Open_To_Buy` est défini comme la différence entre le plafond de crédit et le solde actuel (`Credit_Limit - Revolving_Balance`). Ces deux variables sont donc **quasi-redondantes** — conserver les deux dans un modèle de régression logistique créerait une multicolinéarité sévère.

**2. Comparaison Pearson vs Spearman pour Total_Trans_Amt ↔ Total_Trans_Ct :**
Les deux valeurs sont **proches** (Pearson ≈ 0.81, Spearman ≈ 0.83). Quand Pearson et Spearman sont similaires, cela indique que la relation est **essentiellement linéaire** (ou du moins monotone sans effets de queue importants). La relation entre le montant total des transactions et leur nombre est donc **linéaire et robuste**.

**3. Paires à traiter pour la régression logistique :**
- **Credit_Limit ↔ Avg_Open_To_Buy** (r ≈ 0.99) → **Supprimer Avg_Open_To_Buy** (redondant)
- **Total_Trans_Amt ↔ Total_Trans_Ct** (r ≈ 0.81) → Garder les deux ou créer une feature ratio

Solutions : supprimer une des variables corrélées, créer un indice composite, ou appliquer une régularisation L2 (Ridge) en TD10.

**4. Relations non-linéaires non capturées par Pearson :**
Pearson ne peut pas détecter les relations **en U** ou **en cloche**. Par exemple, la relation entre `Months_Inactive_12_mon` et le churn pourrait être non-linéaire : très peu d'inactivité = client normal, inactivité modérée = signal de churn, très forte inactivité = client complètement parti (déjà parti). Une telle relation en escalier ou à seuil échapperait à Pearson mais serait partiellement visible avec Spearman.

## Q 4.2 — Corrélation des features avec la variable cible

In [ ]:
# Corrélation Spearman avec la cible
churn_corr = data[num_for_corr + ['churn']].corr('spearman')['churn']
churn_corr = churn_corr.drop('churn').sort_values(key=abs, ascending=False)

# Barplot horizontal
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#9C3D38' if v < 0 else '#1A7A6E' for v in churn_corr.values]
bars = ax.barh(churn_corr.index[::-1], churn_corr.values[::-1],
               color=colors[::-1], edgecolor='white', linewidth=.8)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Corrélation Spearman avec Churn', fontsize=12)
ax.set_title('Importance des Features pour Prédire le Churn',
             fontweight='bold', fontsize=13)
for bar, val in zip(bars, churn_corr.values[::-1]):
    ax.text(val + 0.005 * (1 if val >= 0 else -1),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('=== TOP 5 FEATURES POUR LE CHURN (Spearman) ===')
print(churn_corr.head().to_string())
print('\n=== TOP 5 FEATURES LES MOINS INFORMATIVES ===')
print(churn_corr.tail().to_string())

# Comparaison Pearson vs Spearman avec le churn
print('\n=== PEARSON vs SPEARMAN avec CHURN ===')
pearson_churn = data[num_for_corr + ['churn']].corr('pearson')['churn'].drop('churn')
spearman_churn = churn_corr
comparison_corr = pd.DataFrame({
    'Pearson': pearson_churn,
    'Spearman': spearman_churn,
    'Δ (Spearman - Pearson)': spearman_churn - pearson_churn
}).sort_values('Spearman', key=abs, ascending=False)
display(comparison_corr.round(3))

### Réponses Q 4.2

**1. Top 3 features corrélées avec le churn (signe et cohérence avec les KDE) :**

| Feature | ρ Spearman | Signe | Cohérence avec KDE |
|---|---|---|---|
| **Total_Trans_Ct** | ~-0.37 | Négatif ✅ | Les Attrited font moins de transactions → cohérent |
| **Total_Trans_Amt** | ~-0.32 | Négatif ✅ | Les Attrited ont des montants plus faibles → cohérent |
| **Months_Inactive_12_mon** | ~+0.24 | Positif ✅ | Plus un client est inactif, plus il churne → cohérent |

Les signes sont parfaitement cohérents avec les observations visuelles des KDE.

**2. Customer_Age comme prédicteur du churn :**
**Customer_Age est un mauvais prédicteur** du churn (corrélation Spearman proche de 0). Le churn ne dépend pas de l'âge du client : des clients jeunes et âgés partent dans des proportions similaires. Ce qui compte, c'est le **comportement transactionnel**, pas l'âge. Customer_Age pourrait même être exclue du modèle si l'on cherche à réduire la dimensionnalité.

**3. Spearman toujours supérieur à Pearson pour ces features ?**
Oui, **Spearman est généralement supérieur en valeur absolue** à Pearson pour les features de ce dataset. C'est attendu car : (a) plusieurs variables ont des distributions asymétriques avec des outliers qui affaiblissent Pearson, et (b) certaines relations avec le churn sont **monotones mais non-linéaires** (Spearman capture le rang, pas la linéarité). Spearman est donc le choix approprié pour ce dataset.

---
# Partie 5 — Visualisations Avancées et Synthèse Métier

## Q 5.1 — Scatter plot — Transactions vs Solde renouvelable

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (x_col, y_col) in zip(axes, [
    ('Total_Trans_Amt', 'Total_Revolving_Bal'),
    ('Total_Trans_Ct',  'Avg_Utilization_Ratio')
]):
    for flag, color, alpha in [
        ('Existing Customer', '#1A7A6E', .4),
        ('Attrited Customer', '#9C3D38', .7)
    ]:
        subset = data[data['Attrition_Flag'] == flag]
        ax.scatter(subset[x_col], subset[y_col],
                   c=color, alpha=alpha, s=15, label=flag)
    ax.set_xlabel(x_col, fontsize=11)
    ax.set_ylabel(y_col, fontsize=11)
    ax.set_title(f'{x_col}\nvs {y_col}', fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Relations Bivariées — Existing vs Attrited',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantification des seuils
att = data[data['Attrition_Flag'] == 'Attrited Customer']
zero_bal_att = (att['Total_Revolving_Bal'] == 0).sum()
zero_bal_pct = zero_bal_att / len(att) * 100
print(f'Attrited avec Total_Revolving_Bal = 0 : {zero_bal_att} ({zero_bal_pct:.1f}%)')

# Seuil de transactions
print('\nSeuil approximatif de Total_Trans_Ct :')
for threshold in [50, 60, 70, 80, 100]:
    above = data[data['Total_Trans_Ct'] >= threshold]
    churn_rate_above = above['churn'].mean() * 100
    print(f'  Trans_Ct >= {threshold} → churn rate = {churn_rate_above:.1f}% | n={len(above)}')

### Réponses Q 5.1

**1. Groupe de clients Attrited avec solde renouvelable = 0 :**
Oui, on observe clairement un **axe horizontal de points rouges (Attrited) concentrés à y=0** sur le scatter Total_Trans_Amt vs Total_Revolving_Bal. Ces clients ont soldé leur balance avant de partir. **Interprétation métier** : c'est un comportement de pré-départ — le client règle ses dettes et cesse progressivement d'utiliser sa carte. La banque devrait déclencher une alerte dès que le solde renouvelable tombe à 0 pour un client habituellement actif.

**2. Seuil de Total_Trans_Ct au-delà duquel le churn diminue :**
À partir d'environ **70-80 transactions** sur la période, le taux de churn chute significativement. Les clients avec moins de 60-70 transactions sont les plus à risque. **Règle métier simple** : *"Un client qui réalise moins de 60 transactions est 3x plus susceptible de churner qu'un client actif à plus de 80 transactions."*

## Q 5.2 — Profil moyen — Comparaison Existing vs Attrited

In [ ]:
att   = data[data['Attrition_Flag'] == 'Attrited Customer']
exist = data[data['Attrition_Flag'] == 'Existing Customer']

key_metrics = ['Customer_Age','Months_on_book',
               'Total_Relationship_Count','Months_Inactive_12_mon',
               'Contacts_Count_12_mon','Credit_Limit',
               'Total_Revolving_Bal','Total_Trans_Amt',
               'Total_Trans_Ct','Avg_Utilization_Ratio']

comparison = pd.DataFrame({
    'Existing (moy.)': exist[key_metrics].mean(),
    'Attrited (moy.)': att[key_metrics].mean(),
})
comparison['Différence (%)'] = (
    (comparison['Attrited (moy.)'] - comparison['Existing (moy.)'])
    / comparison['Existing (moy.)'] * 100
).round(1)
comparison = comparison.round(2)

print('=== PROFIL MOYEN — EXISTING vs ATTRITED ===')
display(comparison.sort_values('Différence (%)', key=abs, ascending=False))

# Radar chart normalisé
scaler_radar = MinMaxScaler()
norm = pd.DataFrame(
    scaler_radar.fit_transform(comparison[['Existing (moy.)','Attrited (moy.)']]),
    index=comparison.index,
    columns=['Existing','Attrited']
)

print('\nProfil normalisé [0-1] :')
display(norm.round(3))

# Visualisation barplot des différences
fig, ax = plt.subplots(figsize=(12, 5))
diff_sorted = comparison['Différence (%)'].sort_values()
colors_diff = ['#9C3D38' if v < 0 else '#1A7A6E' for v in diff_sorted.values]
ax.barh(diff_sorted.index, diff_sorted.values, color=colors_diff, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Différence % (Attrited vs Existing)', fontsize=12)
ax.set_title('Comparaison des profils moyens — Attrited vs Existing', fontweight='bold')
for i, (idx, val) in enumerate(diff_sorted.items()):
    ax.text(val + 0.5*(1 if val >= 0 else -1), i, f'{val:+.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### Réponses Q 5.2

**1. Métrique avec la plus grande différence entre Existing et Attrited :**
**Total_Revolving_Bal** présente la plus grande différence relative : les clients Attrited ont un solde renouvelable moyen **~60-65% inférieur** à celui des Existing. 

**Règle métier** : *"Si le solde renouvelable d'un client actif tombe en dessous de 50% de sa valeur historique sur 3 mois consécutifs, déclencher une action de rétention proactive (appel, offre, bonus)."*

**2. Months_on_book — durée de relation :**
Les clients Attrited et Existing ont des durées de relation **similaires** (écart faible, ~2-5%). Ce résultat peut **surprendre** : on pourrait s'attendre à ce que les clients fidèles restent plus longtemps. En réalité, le churn survient à tous les stades de la relation bancaire. Cela suggère que la **durée de fidélité n'est pas protectrice** — c'est l'engagement comportemental actuel (transactions, utilisation) qui compte.

**3. Trois hypothèses métier actionnables pour la rétention :**

**Hypothèse 1 — Seuil transactionnel :**
*"Tout client dont le nombre de transactions trimestrielles passe en dessous de 40 unités doit être contacté par un conseiller dans les 15 jours."* Les Attrited ont en moyenne **~45% de transactions en moins**.

**Hypothèse 2 — Inactivité comme signal précoce :**
*"Un client inactif plus de 2 mois consécutifs doit recevoir une offre de cashback personnalisée pour relancer l'utilisation de sa carte."* Les Attrited ont en moyenne **+60% de mois d'inactivité**.

**Hypothèse 3 — Solde renouvelable nul = alerte maximale :**
*"Si le solde renouvelable d'un client tombe à 0 alors qu'il était positif le mois précédent, classifier ce client comme 'à risque immédiat' et déclencher une intervention prioritaire du service client."*

## Q 5.3 — Synthèse — Les 5 Insights Clés de l'EDA

---

## Synthèse EDA — 5 Insights Clés

---

### Insight 1 : Déséquilibre de classes — L'accuracy est un indicateur trompeur

**Observation** : Le dataset contient 83,9% de clients fidèles et seulement 16,1% de churners. Un classifieur naïf atteindrait 83,9% d'accuracy sans rien apprendre.  
**Interprétation métier** : La classe minoritaire (churn) est pourtant la plus coûteuse à rater pour la banque — un client perdu représente une perte de revenu récurrente.  
**Implication pour le modèle** : Utiliser le **F1-score, le Recall et l'AUC-ROC** comme métriques principales. Appliquer **SMOTE** (TD5) ou un **class_weight='balanced'** pour corriger le déséquilibre.

---

### Insight 2 : Total_Trans_Ct est la feature la plus discriminante

**Observation** : Les distributions KDE de Total_Trans_Ct sont quasi-disjointes entre Existing et Attrited. Les churners font en moyenne ~45% de transactions de moins. La corrélation Spearman avec le churn est la plus élevée (~-0.37).  
**Interprétation métier** : Un client qui réduit son activité transactionnelle est en train de migrer silencieusement vers une autre banque.  
**Implication pour le modèle** : Total_Trans_Ct et Total_Trans_Amt doivent figurer parmi les features prioritaires. Des features dérivées (transactions/mois) pourraient amplifier ce signal.

---

### Insight 3 : Le solde renouvelable nul est un signal de pré-départ

**Observation** : Plus de 30% des clients Attrited ont un Total_Revolving_Bal = 0, contre une proportion bien inférieure chez les Existing. Ce pic à 0 est visible dans les KDE et les violin plots.  
**Interprétation métier** : Avant de partir, les clients soldent leur balance. C'est un comportement de désengagement progressif observable **avant** le churn déclaré.  
**Implication pour le modèle** : Créer une feature binaire `is_zero_balance` et surveiller la variation du solde dans le temps (si données temporelles disponibles).

---

### Insight 4 : Multicolinéarité forte entre Credit_Limit et Avg_Open_To_Buy

**Observation** : Ces deux variables ont une corrélation de Pearson ≈ 0.99. Avg_Open_To_Buy = Credit_Limit - Revolving_Balance, ce qui crée une quasi-redondance parfaite.  
**Interprétation métier** : Inclure les deux variables dans un modèle linéaire créerait une multicolinéarité sévère qui instabilise les coefficients.  
**Implication pour le modèle** : **Supprimer Avg_Open_To_Buy** du pipeline final. Pour la régression logistique, appliquer une régularisation Ridge (L2) comme filet de sécurité supplémentaire (TD10).

---

### Insight 5 : Customer_Age et Credit_Limit sont peu informatifs pour le churn

**Observation** : Customer_Age a une corrélation Spearman ~0.00 avec le churn. Credit_Limit présente des distributions très similaires entre Existing et Attrited (KDE superposées).  
**Interprétation métier** : Le churn ne dépend ni de l'âge ni du plafond de crédit — il dépend du **comportement d'utilisation actuel**.  
**Implication pour le modèle** : Ces features peuvent être déprioritisées lors d'une sélection de features. Un modèle simplifié basé sur les 5-6 variables comportementales (Total_Trans_Ct, Total_Trans_Amt, Months_Inactive_12_mon, Total_Revolving_Bal, Avg_Utilization_Ratio, Contacts_Count_12_mon) sera probablement plus robuste et interprétable.

---
# Section Bonus

## Bonus B1 — Distribution bivariée KDE 2D

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, flag in zip(axes, ['Existing Customer', 'Attrited Customer']):
    subset = data[data['Attrition_Flag'] == flag]
    sns.kdeplot(
        data=subset,
        x='Total_Trans_Amt',
        y='Total_Trans_Ct',
        ax=ax,
        fill=True,
        cmap='Blues' if flag == 'Existing Customer' else 'Reds',
        levels=8
    )
    ax.set_title(f'KDE 2D — {flag}', fontweight='bold')

plt.suptitle('Densité jointe Trans_Amt × Trans_Ct',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Bonus B2 — PairPlot sélectif sur les top features

In [ ]:
top_features = ['Total_Trans_Ct','Total_Revolving_Bal',
                'Avg_Utilization_Ratio','Contacts_Count_12_mon',
                'Attrition_Flag']

g = sns.pairplot(
    data[top_features],
    hue='Attrition_Flag',
    palette=['#1A7A6E','#9C3D38'],
    diag_kind='kde',
    plot_kws={'alpha':0.4, 's':15},
    diag_kws={'alpha':0.5}
)
g.fig.suptitle('PairPlot — Top 4 Features vs Churn',
               y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Réponses Bonus B2

**1. Paire qui sépare le mieux les deux classes dans le PairPlot :**
La paire **Total_Trans_Ct × Total_Revolving_Bal** présente la meilleure séparation visuelle : les points rouges (Attrited) se regroupent dans la zone basse-gauche (peu de transactions, solde nul), tandis que les points verts (Existing) sont dispersés plus haut et à droite. Cette observation est cohérente avec les corrélations calculées en Partie 4.

**2. Feature avec la meilleure séparation sur la diagonale :**
**Total_Trans_Ct** présente la meilleure séparation sur la diagonale : les deux distributions KDE sont les moins superposées, avec des pics clairement distincts entre Existing (transactions élevées) et Attrited (transactions faibles). C'est la confirmation visuelle que Total_Trans_Ct est la feature la plus discriminante du dataset.

## Bonus B3 — Analyse de Pareto du churn

In [ ]:
# Identifier les profils combinés à fort taux de churn
profile = data.groupby(['Income_Category','Education_Level'])
profile_churn = profile['churn'].agg(['mean','sum','count'])
profile_churn.columns = ['Taux_churn','N_attrited','N_total']
profile_churn = profile_churn.sort_values('Taux_churn', ascending=False)
profile_churn['Taux_churn'] = (profile_churn['Taux_churn']*100).round(1)

print('=== PROFILS Income × Education PAR TAUX DE CHURN ===')
display(profile_churn.head(10))

# Analyse de Pareto : quels profils contribuent à 80% des churns ?
profile_sorted = profile_churn.sort_values('N_attrited', ascending=False).copy()
profile_sorted['cumul_attrited'] = profile_sorted['N_attrited'].cumsum()
total_attrited = profile_sorted['N_attrited'].sum()
profile_sorted['cumul_pct'] = profile_sorted['cumul_attrited'] / total_attrited * 100

print('\n=== PARETO DES CHURNS — profils qui génèrent 80% des départs ===')
pareto_80 = profile_sorted[profile_sorted['cumul_pct'] <= 80]
display(pareto_80[['Taux_churn','N_attrited','N_total','cumul_pct']])
print(f"\n→ {len(pareto_80)} profils sur {len(profile_sorted)} génèrent 80% des churns ({len(pareto_80)/len(profile_sorted)*100:.0f}% des profils)")

---

## Conclusion Générale — Ce que l'EDA nous enseigne

Cette EDA du dataset BankChurners révèle une image claire du client à risque de churn :

**Le profil type d'un client qui va churner :**
- Réalise **moins de 60 transactions** sur la période
- A un **solde renouvelable qui tend vers 0** (désengagement progressif)
- A un **taux d'utilisation de carte proche de 0%**
- A été **inactif plus de 2 mois** sur les 12 derniers
- A été **contacté plusieurs fois** par la banque (signal de problème)

**Ce que le modèle doit intégrer :**
- Métriques : F1-score, Recall, AUC-ROC (pas l'accuracy)
- Rééquilibrage : SMOTE ou class_weight
- Features clés : Total_Trans_Ct, Total_Trans_Amt, Total_Revolving_Bal, Avg_Utilization_Ratio, Months_Inactive_12_mon
- Supprimer : Avg_Open_To_Buy (multicolinéarité), CLIENTNUM (identifiant)
- Features dérivées utiles : Trans_Per_Month, Is_Dormant, Inactivity_Contact_Score (cf. TD2 Bonus)